In [1]:
import opensim as osim

In [2]:
# Define the global model
arm = osim.Model()

In [3]:
# Create humerus aand radius bones as OpenSim bodies
humerus = osim.Body(
    "humerus",  # name
    0.5,  # mass
    osim.Vec3(0),  # center of mass
    osim.Inertia(0.01, 0.01, 0.01, 0, 0, 0),
)

radius = osim.Body(
    "radius",  # name
    0.3,  # mass
    osim.Vec3(0),  # center of mass
    osim.Inertia(0.01, 0.01, 0.01, 0, 0, 0),
)

# Connect the bodies with a pin joint
shoulder = osim.PinJoint(
    "shoulder",
    arm.getGround(),  # PhysicalFrame
    osim.Vec3(0, 0, 0),
    osim.Vec3(0, 0, 0),
    humerus,  # PhysicalFrame
    osim.Vec3(0, 1, 0),
    osim.Vec3(0, 0, 0),
)

elbow = osim.PinJoint(
    "elbow",
    humerus,  # PhysicalFrame
    osim.Vec3(0, 0, 0),
    osim.Vec3(0, 0, 0),
    radius,  # PhysicalFrame
    osim.Vec3(0, 1, 0),
    osim.Vec3(0, 0, 0),
)

In [4]:
# Add a muscle that flexes the elbow (actuator for robotics people).
biceps = osim.Millard2012EquilibriumMuscle(
    "biceps",  # Muscle name
    200.0,  # Max isometric force
    0.6,  # Optimal fibre length
    0.55,  # Tendon slack length
    0.0,
)  # Pennation angle
biceps.addNewPathPoint("origin", humerus, osim.Vec3(0, 0.8, 0))

biceps.addNewPathPoint("insertion", radius, osim.Vec3(0, 0.7, 0))

In [5]:
# Add a controller that specifies the excitation of the muscle.
brain = osim.PrescribedController()
brain.addActuator(biceps)
brain.prescribeControlForActuator("biceps", osim.StepFunction(0.5, 3.0, 0.3, 1.0))

In [6]:
# Build model with components created above.
arm.addBody(humerus)
arm.addBody(radius)
arm.addJoint(shoulder)  # Now required in OpenSim4.0
arm.addJoint(elbow)
arm.addForce(biceps)
arm.addController(brain)

In [7]:
# Add a console reporter to print the muscle fibre force and elbow angle.
# We want to write our simulation results to the console.
reporter = osim.ConsoleReporter()
reporter.set_report_time_interval(1.0)
reporter.addToReport(biceps.getOutput("fiber_force"))
elbow_coord = elbow.getCoordinate().getOutput("value")
reporter.addToReport(elbow_coord, "elbow_angle")
arm.addComponent(reporter)

In [8]:
# Add display geometry.
bodyGeometry = osim.Ellipsoid(0.1, 0.5, 0.1)
bodyGeometry.setColor(osim.Gray)
humerusCenter = osim.PhysicalOffsetFrame()
humerusCenter.setName("humerusCenter")
humerusCenter.setParentFrame(humerus)
humerusCenter.setOffsetTransform(osim.Transform(osim.Vec3(0, 0.5, 0)))
humerus.addComponent(humerusCenter)
humerusCenter.attachGeometry(bodyGeometry.clone())

radiusCenter = osim.PhysicalOffsetFrame()
radiusCenter.setName("radiusCenter")
radiusCenter.setParentFrame(radius)
radiusCenter.setOffsetTransform(osim.Transform(osim.Vec3(0, 0.5, 0)))
radius.addComponent(radiusCenter)
radiusCenter.attachGeometry(bodyGeometry.clone())

In [9]:
# Configure the model.
state = arm.initSystem()
# Fix the shoulder at its default angle and begin with the elbow flexed.
shoulder.getCoordinate().setLocked(state, True)
elbow.getCoordinate().setValue(state, 0.5 * osim.SimTK_PI)
arm.equilibrateMuscles(state)

In [10]:
# Simulate.
manager = osim.Manager(arm)
state.setTime(0)
manager.initialize(state)
state = manager.integrate(10.0)

In [11]:
# Print/save model file
arm.printToXML("SimpleArm.osim")

True